# Atividade em Python — Solução regular
**Termodinâmica Avançada — 2026.2 · Aula 10**

Cinco exercícios sobre a mesma equação da aula:

$$\frac{\Delta g_{\mathrm{mist}}}{RT}
= \underbrace{x\ln x+(1-x)\ln(1-x)}_{\text{entropia da mistura ideal}}
\;+\;\underbrace{\xi\,x(1-x)}_{\text{entalpia de excesso}}$$

com $\xi = w/RT$. **Cada exercício termina num `assert` com o número que
apareceu na tela da aula**: se a célula imprimir `ok`, está certo — não é
preciso esperar correção.

Entregue este caderno executado, ao final da aula.

> Uso de IA liberado para ajudar no código.

## Preparo
Rode esta célula primeiro.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import brentq

R = 8.314          # J/(mol K)

# O parâmetro da solução regular tem duas formas, e as duas aparecem na aula:
#   w   (J/mol)      o preço de trocar contatos A-A e B-B por A-B; NÃO depende de T
#   ξ = w/(R T)      a forma adimensional, que é a dos slides e a do Atkins
def xi_de_w(w, T):
    return w / (R * T)

print(f"R = {R} J/(mol K)")

## 1. As curvas, de $\xi=-3$ a $\xi=3$

Escreva `gmix(x, xi)` e trace a família de curvas para
$\xi \in \{-3,-1,0,1,1{,}15,2,2{,}5,3\}$.

**Esperado:** em $x=0{,}5$, a curva de $\xi=0$ vale $\ln 0{,}5 = -0{,}6931$
(é a mistura ideal da Aula 09), a de $\xi=1{,}15$ (benzeno + ciclo-hexano a
$25\ ^\circ\text{C}$) vale $-0{,}4056$, a de $\xi=2$ vale $-0{,}1931$ e a de
$\xi=3$ já é **positiva**, $+0{,}0569$.

Olhe o fundo de cada curva enquanto $\xi$ cresce.

In [ ]:
def gmix(x, xi):
    """Δg_mist/(RT) da solução regular binária."""
    # SEU CÓDIGO AQUI --- cuide do log(0) com np.clip
    raise NotImplementedError

x = np.linspace(1e-6, 1 - 1e-6, 600)
fig, ax = plt.subplots(figsize=(7, 4.6))
for xi in (-3, -1, 0, 1, 1.15, 2, 2.5, 3):
    ax.plot(x, gmix(x, xi), label=f"ξ = {xi}")
ax.axhline(0, color="k", lw=0.6)
ax.set_xlabel("$x$"); ax.set_ylabel(r"$\Delta g_{mist}/RT$")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

for xi in (0, 1.15, 2, 3):
    print(f"ξ = {xi}:  Δg/RT em x = 0,5  ->  {gmix(0.5, xi):+.4f}")

assert abs(gmix(0.5, 0) - np.log(0.5)) < 1e-9
assert abs(gmix(0.5, 1.15) - (-0.4056)) < 1e-3
assert abs(gmix(0.5, 2) - (-0.1931)) < 1e-3
assert abs(gmix(0.5, 3) - (+0.0569)) < 1e-3
print("ok")

## 2. Onde nasce a lacuna

Os dois mínimos saem de derivada nula:

$$\ln\frac{x}{1-x} + \xi\,(1-2x) = 0 .$$

Escreva `coex(xi)` que devolve as duas composições — ou `None` quando não
houver lacuna. Resolva no ramo $x<0{,}5$ com `brentq` e use a simetria para
o outro.

**Esperado:** nada até $\xi=2$, e em $\xi=2{,}5$ as composições
$0{,}1448$ e $0{,}8552$ — os números do vídeo.

In [ ]:
def coex(xi):
    """As duas composições em coexistência, ou None se não houver lacuna."""
    # SEU CÓDIGO AQUI: devolva None se xi <= 2; senão resolve
    #   ln[x/(1-x)] + xi(1-2x) = 0
    # no ramo x < 0,5 com brentq, e use a simetria para o outro ramo.
    raise NotImplementedError

for xi in (1.9, 2.0, 2.1, 2.5, 3.0):
    r = coex(xi)
    print(f"ξ = {xi:4.1f}  ->  " +
          ("sem lacuna" if r is None else f"x = {r[0]:.4f} e {r[1]:.4f}"))

assert coex(2.0) is None
assert coex(2.0001) is not None
xa, xb = coex(2.5)
assert abs(xa - 0.1448) < 1e-3 and abs(xb - 0.8552) < 1e-3
print("ok")

## 3. A tangente comum, e por que a mistura se separa

Trace a curva de $\xi=2{,}5$ com a reta que liga os dois mínimos, e marque
em $x=0{,}5$ **dois** pontos: o da curva (a mistura homogênea) e o da reta
(a mistura separada em metade de cada fase).

**Esperado:** $-0{,}0681$ para a homogênea contra $-0{,}1040$ para a
separada. A reta está mais baixo — e é por isso que o frasco se separa.

In [ ]:
xi = 2.5
xa, xb = coex(xi)
ga, gb = gmix(xa, xi), gmix(xb, xi)

# SEU CÓDIGO AQUI: trace a curva, marque os dois mínimos ligados por uma
# reta (a tangente comum), e marque os dois pontos em x = 0,5 --- o da
# curva (mistura homogênea) e o da reta (mistura separada).
raise NotImplementedError

homogenea = gmix(0.5, xi)
separada = 0.5*(ga + gb)
print(f"homogênea em x = 0,5:        {homogenea:+.4f}")
print(f"separada, metade de cada:    {separada:+.4f}")
print(f"ganho ao separar:            {separada - homogenea:+.4f}")

assert separada < homogenea
assert abs(homogenea - (-0.0681)) < 1e-3
assert abs(separada - (-0.1040)) < 1e-3
print("ok")

## 4. O domo, com um $w$ medido

Agora fixe $w$ e varie a temperatura, lembrando que $\xi = w/RT$. Use
$w = 5639$ J/mol, que é o valor tirado da temperatura crítica **medida** da
mistura água + fenol (66 °C, Oliveira Fig. 10.6).

Monte o diagrama $T\times x$: para cada $T$ abaixo de $T_c$, marque as duas
composições de `coex`. O resultado é o *domo* de imiscibilidade.

**Esperado:** $T_c = w/2R = 339{,}15$ K $= 66$ °C, e nenhuma lacuna acima
disso.

In [ ]:
w = 5639.0                      # J/mol, de T_c = 66 °C medida (água + fenol)
# SEU CÓDIGO AQUI:
#   1. calcule T_c = w/(2R) e imprima em K e em °C;
#   2. varra T abaixo de T_c, obtenha coex(xi_de_w(w, T)) e guarde os ramos;
#   3. trace o domo (x no eixo horizontal, temperatura no vertical) e marque
#      o ponto crítico.
raise NotImplementedError

assert abs(T_c - 339.15) < 0.05
assert coex(xi_de_w(w, T_c + 1.0)) is None      # acima de T_c, sem lacuna
assert coex(xi_de_w(w, T_c - 1.0)) is not None  # abaixo, com lacuna
print("ok")

## 5. Ligas de alta entropia

Em 2004 Yeh e colaboradores sintetizaram as primeiras *ligas de alta
entropia*; meses depois Cantor e colaboradores fizeram a **CrMnFeCoNi** —
cinco metais em frações iguais, numa solução sólida única.

Por que cinco componentes ajudam? Compare os dois termos para $n$
componentes equimolares:

- entropia: $\Delta s = R\ln n$ — cresce **sem teto**;
- entalpia: se todos os pares tiverem o mesmo $w$, $\Delta h = w\,\sum_{i<j}x_ix_j$,
  que com $x_i = 1/n$ dá $w\,(n-1)/2n$ — e isso **satura em $w/2$**.

Calcule os dois para $n = 2$ a $7$.

**Esperado:** $R\ln 5 = 13{,}381$ J/(mol K), e o termo de entalpia nunca
passa de $w/2$. É essa corrida desigual que estabiliza a liga.

In [ ]:
def ds_equimolar(n):
    """Δs de mistura de n componentes em frações iguais, J/(mol K)."""
    # SEU CÓDIGO AQUI
    raise NotImplementedError

def h_equimolar(n, w):
    """Δh de mistura, J/mol, se TODOS os pares tiverem o mesmo w.

    Dica: Δh = w · Σ_{i<j} x_i x_j, e com n frações iguais a 1/n existem
    n(n-1)/2 pares.
    """
    # SEU CÓDIGO AQUI
    raise NotImplementedError

w = 5639.0
print(f"{'n':>3} {'Δs = R ln n':>14} {'Δh':>12}")
for n in range(2, 8):
    print(f"{n:3d} {ds_equimolar(n):14.3f} {h_equimolar(n, w):12.1f}")
print(f"\nteto do termo de entalpia: w/2 = {w/2:.1f} J/mol")

assert abs(ds_equimolar(5) - 13.381) < 1e-3
assert h_equimolar(7, w) < w/2
assert ds_equimolar(7) > ds_equimolar(2)
print("ok")

## Para entregar

Este caderno executado, com as cinco células imprimindo `ok`, e as duas
respostas abaixo escritas numa célula de texto.

1. **Qual o significado físico de $\xi_c = 2$?** Responda em termos da
   competição entre os dois termos, e diga o que $\xi = 2$ quer dizer sobre
   a temperatura, usando $\xi = w/RT$.

2. No exercício 4 o domo saiu **simétrico** em torno de $x = 0{,}5$. As
   composições medidas das duas fases da água + fenol a 20 °C são
   $0{,}016$ e $0{,}330$ em fração molar. O modelo errou — mas errou em quê, exatamente? O que ele
   ainda acerta, e o que seria preciso mudar nele para acertar o resto?